# 02 — EDA and Filtering

In this notebook, we explore the cleaned Steam interaction data and decide which user-game interactions should be used for the similarity analysis.

The main goal is to distinguish between simply owning a game and actually engaging with it.

In [3]:
from pathlib import Path

import pandas as pd


DATA_PATH = Path("../data/processed/interactions_clean.parquet")

interactions = pd.read_parquet(DATA_PATH)

print("Interactions:", len(interactions))
print("Users:", interactions["user_id"].nunique())
print("Games:", interactions["game_id"].nunique())

interactions.head()

Interactions: 5094082
Users: 70912
Games: 10978


,user_id,game_id,game_name,playtime_forever,playtime_2weeks
0,wolop,4000,Garry's Mod,642773,7299
1,Evilutional,212200,Mabinogi,635295,0
2,76561198019826668,42710,Call of Duty: Black Ops - Multiplayer,632452,8126
3,76561198039832932,4000,Garry's Mod,613411,588
4,tsunamitad,72200,Universe Sandbox,600068,0


## Explore playtime

Many Steam users own games they have never played. We inspect lifetime playtime to see how much data remains under different engagement thresholds.

In [4]:
# Compare how many interactions remain at different playtime thresholds.
thresholds = {
    "Owned": 0,
    "Played > 0 min": 1,
    "Played >= 2 hours": 120,
    "Played >= 5 hours": 300,
    "Played >= 10 hours": 600
}

for label, minutes in thresholds.items():
    count = (interactions["playtime_forever"] >= minutes).sum()
    percentage = count / len(interactions) * 100

    print(f"{label}: {count:,} interactions ({percentage:.1f}%)")

Owned: 5,094,082 interactions (100.0%)
Played > 0 min: 3,246,352 interactions (63.7%)


Played >= 2 hours: 1,941,641 interactions (38.1%)
Played >= 5 hours: 1,373,990 interactions (27.0%)
Played >= 10 hours: 972,626 interactions (19.1%)


## Explore game popularity

Very rare games may create unreliable similarities because only a few users have played them.

We therefore inspect how many engaged players each game has.

In [5]:
# Start with a 2-hour engagement threshold.
engaged = interactions[
    interactions["playtime_forever"] >= 120
]

players_per_game = (
    engaged
    .groupby("game_id")["user_id"]
    .nunique()
)

print(players_per_game.describe())

count     8342.000000
mean       232.754855
std       1235.257985
min          1.000000
25%          4.000000
50%         15.000000
75%         64.000000
max      40887.000000
Name: user_id, dtype: float64


In [6]:
# See how many games remain at different minimum player counts.
minimum_players = [10, 25, 50, 100]

for minimum in minimum_players:
    games_remaining = (players_per_game >= minimum).sum()

    print(
        f"At least {minimum} players: "
        f"{games_remaining:,} games"
    )

At least 10 players: 4,951 games
At least 25 players: 3,360 games
At least 50 players: 2,402 games
At least 100 players: 1,634 games


## Create the analysis dataset

For the similarity analysis, we define an engaged player as someone with at least 2 hours of playtime.

We also keep only games with at least 50 engaged players to avoid similarities based on very small player groups.

In [7]:
# Keep games with at least 50 engaged players.
valid_games = players_per_game[
    players_per_game >= 50
].index

interactions_filtered = engaged[
    engaged["game_id"].isin(valid_games)
].copy()

print("Interactions:", len(interactions_filtered))
print("Users:", interactions_filtered["user_id"].nunique())
print("Games:", interactions_filtered["game_id"].nunique())

Interactions: 1871351
Users: 66754
Games: 2402


In [8]:
FILTERED_PATH = Path("../data/processed/interactions_filtered.parquet")

interactions_filtered.to_parquet(
    FILTERED_PATH,
    index=False
)

print("Saved filtered interactions to:")
print(FILTERED_PATH)
print("Shape:", interactions_filtered.shape)

Saved filtered interactions to:
../data/processed/interactions_filtered.parquet
Shape: (1871351, 5)


## Summary

For the similarity analysis, we use:

- At least **2 hours of playtime** to count a user as engaged with a game
- At least **50 engaged players** for a game to be included

This leaves:

- **1,871,351 interactions**
- **66,754 users**
- **2,402 games**

The filtered dataset is saved as:

`data/processed/interactions_filtered.parquet`